<a href="https://colab.research.google.com/github/pedroManuelP/C-digos-em-Python/blob/main/SistControle_C%C3%B3digoProva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import numpy as np
from numpy.polynomial import Polynomial
import scipy as sp
import scipy.signal as signal
import re

def FT_to_EE(g_num, g_den, s):
  # realização da FT para EE
  n = g_den.degree();
  alpha = g_den.coef[0:-1]; beta = g_num.coef;
  if(s == '1'):
    # forma canônica controlável
    A = np.identity(n); A = np.roll(A, -1, axis = 0); A[-1][0] = 0; A[-1,:] = -alpha
    B = np.zeros((n, 1)); B[-1, 0] = 1
    C = np.zeros((1, n)); C[0,:] = beta
  if(s == '2'):
    # forma canônica observável
    A = np.identity(n); A = np.roll(A, -1, axis = 1); A[0][-1] = 0; A[:, -1] = -alpha
    B = np.zeros((n,1)); B[:,0] = beta.T
    C = np.zeros((1,n)); C[0, -1] = 1
  return A, B, C

def cont2disc(A, B, C, T):
  sys_c = sp.signal.StateSpace(A, B, C);
  sys_d = sys_c.to_discrete(T);
  return np.round(sys_d.A, 3), np.round(sys_d.B, 3), np.round(sys_d.C, 3)

def analiseEstabilidade(A, B, C, disc):
  # Analisar estabilidade
  A = -A; s = Polynomial([0, 1])
  if(A.shape[0] == 3):
    plus = (s+A[0][0])*(s+A[1][1])*(s+A[2][2]) + (A[0][1]*A[1][2]*A[2][0]) + (A[0][2]*A[1][0]*A[2][1])
    minus = (A[2][0]*(s+A[1][1])*A[0][2]) + (A[2][1]*A[1][2]*(s+A[0][0])) + ((s+A[2][2])*A[1][0]*A[0][1])
    poly = plus-minus; polos = poly.roots()
  elif(A.shape[0] == 2):
    plus = (s+A[0][0])*(s+A[1][1])
    minus = (A[1][0]*A[0][1])
    poly = plus-minus; polos = poly.roots()
  elif(A.shape[0] == 1):
    poly = s+A[0][0]; polos = poly.roots()
  else:
    print("Não é possível calcular o polinômio característico.")

  # polos do polinômio característico
  polos = np.round(polos, 3)
  print("Polos: " + str(polos))

  # Analise de estabilidade para contínuo e discreto
  if (disc == '1'):
    # verifica se algum polo está fora do círculo unitário
    for i in range(len(polos)):
      if(np.abs(polos[i].real) > 0):
        print("O sistema é instável.")
        return
    print("O sistema é estável.")
  if(disc == '2'):
    # verifica se algum polo tem parte real negativa
    for i in range(len(polos)):
      if(polos[i].real > 0):
        print("O sistema é instável.")
        return
    print("O sistema é estável.")

def matrizControlabilidade(A, B):
  n = A.shape[0]
  U = np.zeros((n,n))
  for i in range(n):
    U[:,i:i+1] = (np.linalg.matrix_power(A,i))@B

  if(np.linalg.matrix_rank(U) == n):
    return U, True
  else:
    return U, False

def matrizObservabilidade(A, C):
  n = A.shape[0]
  V = np.zeros((n,n))
  for i in range(n):
    V[i:i+1, :] = C@(np.linalg.matrix_power(A,i))

  if(np.linalg.matrix_rank(V) == n):
    return V, True
  else:
    return V, False

def realimentadorDeEstado(A, U, s):
  theta = Polynomial.fromroots(s)
  coeficientes = (theta.coef).real

  n = A.shape[0]; qc = np.zeros((n,n))
  for i in range(n+1):
    qc += coeficientes[i]*(np.linalg.matrix_power(A,i))
  invU = np.linalg.inv(U)
  linha = np.zeros((1,n)); linha[0][-1] = 1
  return -linha@(invU@qc)

def observadorDeEstado(A, V, s):
  theta = Polynomial.fromroots(s)
  coeficientes = (theta.coef).real

  n = A.shape[0]; ql = np.zeros((n,n))
  for i in range(n+1):
    ql += coeficientes[i]*(np.linalg.matrix_power(A,i))
  invV = np.linalg.inv(V)
  coluna = np.zeros((n,1)); coluna[-1][0] = 1
  return ql@invV@coluna

def matrizExpandida(A, B, C, disc):
  # Matriz expandida para um seguidor de referência do tipo degrau unitário
  if(disc == '2'):
    # Sistema contínuo
    n = np.shape(A)[0]; Ae = np.zeros((n+1, n+1));
    Ae[0, 1:(n+1)] = C; Ae[1:(n+1), 1:(n+1)] = A
    Be = np.zeros(((n+1), 1)); Be[1:(n+1), 0] = B[:, 0]
    print("Ae = "+str(Ae)); print("Be = "+str(Be));
  elif(disc == '1'):
    # Sistema discreto
    n = np.shape(A)[0]; Ae = np.zeros((n+1, n+1));
    Ae[0:n, 0:n] = A; Ae[0:n, n:n+1] = B;
    Be = np.zeros((n+1, 1)); Be[-1, 0] = 1;
    print("Ge = "+str(Ae)); print("He = "+str(Be));
  return Ae, Be

def seguidorReferência(Ae, Be, Wce, polos):
  Ke = realimentadorDeEstado(Ae, Wce, polos);
  if(disc == '1'):
    n = np.shape(A)[0]; a = np.zeros([1,n+1]); a[0,-1] = 1;

    b = np.zeros((n+1, n+1));
    b[0:n, 0:n] = A-np.eye(n); b[0:n, n:n+1] = B;
    b[n, 0:n] = np.matmul(C,A); b[n, n:n+1] = np.matmul(C,B);
    return np.round(np.matmul((Ke+a),np.linalg.inv(b)), 3)
  if(disc == '2'):
    return Ke

def input_matriz(nome):
  print(f"\nDigite a matriz {nome} (separando elementos por espaços, linhas por ponto e vírgula):")
  print(f"Exemplo: 0 1; -2 -3")
  s = input().strip()
  linhas = s.split(';')
  matriz = []
  for linha in linhas:
      # .strip() remove os whitespaces apenas do início e fim das strings
      # .split() "corta" a string em um lista considerando a vírgula como um separador
      elementos = [float(x) for x in linha.strip().split()]

      # toda vez que .append é executado, uma nova linha é criada
      matriz.append(elementos)
  return np.array(matriz)

def ler_polos():
    print("Digite os polos desejados (um por linha, Enter vazio para terminar):")
    print("Para polos complexos, digite: -0.5 0.5 (parte real parte imaginária)")
    polos = []
    while True:
        s = input().strip()
        if s == '':
            break
        valores = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
        if len(valores) == 2:
            polos.append(complex(valores[0], valores[1]))
        elif len(valores) == 1:
            polos.append(complex(valores[0], 0))
        else:
            print("Entrada inválida. Tente novamente.")
    return np.array(polos)


In [32]:
# Escolha do método de entrada
print("="*50); print('');
print("Como deseja definir o sistema?")
print("1 - Pela função de transferência G(s)")
print("2 - Inserindo as matrizes A, B, C")
opcao = input("Digite 1 ou 2: ").strip()

# Definição do sistema
if opcao == '1':
    # Função G(s)
    print("\nCoeficientes do numerador de G(s): ", end = ''); s = input()
    coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
    g_num = Polynomial(np.flip(coef_arr))
    print("Coeficientes do denominador de G(s): ", end = ''); s = input()
    coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
    g_den = Polynomial(np.flip(coef_arr))

    # Impressão das funções
    print("G(x) = (" + str(g_num) + ")/(" + str(g_den) + ")\n")

    # Representação dos Espaços de Estados
    print("Como desejar representar a função no espaço de estados?")
    print("Digite 1 para forma canônica controlável;"); print("Digite 2 para forma canônica observável.")
    s = input(); A, B, C = FT_to_EE(g_num, g_den, s)

elif opcao == '2':
    print("\n--- Entrada direta das matrizes de espaço de estados ---")

    # Entrada da matriz A
    A = input_matriz("A")

    # Entrada da matriz B
    print(f"\nMatriz A tem dimensão {A.shape[0]}x{A.shape[1]}")
    print(f"A matriz B deve ter {A.shape[0]} linhas")
    B = input_matriz("B")

    # Verificar dimensão de B
    if B.shape[0] != A.shape[0]:
        print(f"Erro: B deve ter {A.shape[0]} linhas. Reinicie o programa.")
        exit()

    # Entrada da matriz C
    print(f"\nA matriz C deve ter {A.shape[1]} colunas")
    C = input_matriz("C")

    # Verificar dimensão de C
    if C.shape[1] != A.shape[1]:
        print(f"Erro: C deve ter {A.shape[1]} colunas. Reinicie o programa.")
        exit()

    # Se C for um vetor linha, garantir o formato correto
    if len(C.shape) == 1:
        C = C.reshape(1, -1)

    print("\nSistema definido pelas matrizes:")

else:
    print("Opção inválida! Reinicie o programa.")
    exit()

# Impressão das matrizes do sistema
print("A = " + str(A))
print("B = " + str(B))
print("C = " + str(C))

# Discretização do sistema
print(); print("="*50); print('');
print("Deseja discretizar o sistema?")
print("1 - Sim")
print("2 - Não")
disc = input("Escolha (1 ou 2): ").strip()

if disc == '1':
    T = float(input("Período de amostragem T (s): ").strip());
    G, H, C = cont2disc(A, B, C, T);
    print("Sistema discretizado:")
    print("G = " + str(G))
    print("H = " + str(H))
    print("C = " + str(C))
    A, B, C = G, H, C
else:
    print("Sistema não discretizado.")

# Análise completa do sistema
print(''); print("="*50); print('');
print("Análise completa do sistema:")
analiseEstabilidade(A, B, C, disc);

[Wc,controlabilidade] = matrizControlabilidade(A, B);
print("\nMatriz de Controlabilidade:"); print(Wc)
if(controlabilidade):
    print("Sistema é controlável.")
else:
    print("Sistema não é controlável.")

[Wo,observabilidade] = matrizObservabilidade(A, C);
print("\nMatriz de Observabilidade:"); print(Wo)
if(observabilidade):
    print("Sistema é observável.")
else:
    print("Sistema não é observável.")

# Projeto do realimentador de estados
print(''); print("="*50); print('');
if(controlabilidade):
  print("O sistema é controlável. É possível projetar um realimentador de estados.")
  polos = ler_polos();
  if len(polos) > 0:
    K = realimentadorDeEstado(A, Wc, polos)
    print("Realimentador de estados:")
    print("K = " + str(K))
  else:
      print("Nenhum polo informado.")
else:
  print("O sistema não é controlável. É não é possível projetar um realimentador de estados.")

# Projeto do observador de estados
print(''); print("="*50); print('');
if(observabilidade):
  print("O sistema é observável. É possível projetar um observador de estados.")
  polos = ler_polos();
  if len(polos) > 0:
    L = observadorDeEstado(A, Wo, polos)
    print("Observador de estados:")
    print("L = " + str(L))
  else:
      print("Nenhum polo informado.")
else:
  print("O sistema não é observável. É não é possível projetar um observador de estados.")

# Projeto do seguidor de referência
print(''); print("="*50); print('');
[Ae, Be] = matrizExpandida(A,B, C, disc);
[Wce, controlabilidade] = matrizControlabilidade(Ae, Be); print("\nWce = " + str(Wce));
if(controlabilidade):
  print("O sistema expandido é controlável. É possível projetar um seguidor de referência."); print()
  polos = ler_polos();
  if len(polos) > 0:
    Kr = seguidorReferência(Ae, Be, Wce, polos);
    print("Seguidor de referência:")
    print("K = " + str(Kr))
    if(disc == '2'):
      print("k1 = "+str(Kr[0][0])); print("k2 = "+str(Kr[0][1:Kr.shape[1]]))
    elif(disc == '1'):
      print("k1 = "+str(Kr[0][-1])); print("k2 = "+str(Kr[0][0:(Kr.shape[1]-1)]));
  else:
    print("Nenhum polo informado.")
else:
  print("O sistema expandido não é controlável. É não é possível projetar um seguidor de referência.")


Como deseja definir o sistema?
1 - Pela função de transferência G(s)
2 - Inserindo as matrizes A, B, C
Digite 1 ou 2: 1

Coeficientes do numerador de G(s): 3 -2
Coeficientes do denominador de G(s): 1 1 0
G(x) = (-2.0 + 3.0·x)/(0.0 + 1.0·x + 1.0·x²)

Como desejar representar a função no espaço de estados?
Digite 1 para forma canônica controlável;
Digite 2 para forma canônica observável.
1
A = [[ 0.  1.]
 [-0. -1.]]
B = [[0.]
 [1.]]
C = [[-2.  3.]]


Deseja discretizar o sistema?
1 - Sim
2 - Não
Escolha (1 ou 2): 1
Período de amostragem T (s): 1
Sistema discretizado:
G = [[1.    0.632]
 [0.    0.368]]
H = [[0.368]
 [0.632]]
C = [[-2.  3.]]


Análise completa do sistema:
Polos: [0.368 1.   ]
O sistema é instável.

Matriz de Controlabilidade:
[[0.368    0.767424]
 [0.632    0.232576]]
Sistema é controlável.

Matriz de Observabilidade:
[[-2.    3.  ]
 [-2.   -0.16]]
Sistema é observável.


O sistema é controlável. É possível projetar um realimentador de estados.
Digite os polos desejados (